<a id="unit-u.331fee2e3573454898fb0fa91eab188b"></a>

<div>
  <div>12-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Көріністі рөлге жариялап, кім не оқығанын аудиттеу</div>
</div>

11-модуль `bi_order_metrics_l2` жариялады, BI тобы тіркелгі сұрайды. Рөлге тек осы көріністі оқуға рұқсат беріп, уақытша қарапайым пайдаланушы ретінде көрініс оқылатынын, дереккөздері оқылмайтынын дәлелдейсіз, қайта тоқтатасыз. 11-модульдің қосымша тақтасына тікелей рұқсат берілген тіркелгіні тауып, рөлге көшіресіз. Кейін тексерілген ауысу мен кері қайтаруы бар тұрақты атаумен кесте жариялап, екі аймақ рөлін өз тапсырыстарымен Row Policy арқылы шектейсіз; үшіншісінен төлем бағандарын баған рұқсатымен жасырасыз; қарапайым пайдаланушылар сұрауларын, тыйымды да Doris аудитінен табасыз. Шамамен 75 минут жоспарлаңыз.

**12-модуль: Жариялау, рұқсаттар және аудит** · [Сабақ](course.md) · [Тест](quiz12_publishing_permissions_audit.ipynb) · [Курстың README файлы](../../README.md)

<a id="unit-u.a4ae443254634381a13825daef0a1cd6"></a>

<a id="before-you-begin"></a>

## Бастамас бұрын

Алдымен бір базада 9 және 11-жұмыстарды аяқтаңыз. Бұл жұмыс `bi_order_metrics_l2` жариялап, 10-жұмыстың `daily_order_metrics_l2`, 9-жұмыстың `orders_fact_l2` дереккөздеріне қатынауды тексереді; ешқайсысын өзгертпейді. Біреуі жоқ болса, бастапқы ұяшық қай жұмысты аяқтауды айтып тоқтайды.

Рөлдер, пайдаланушылар, Row Policy бүкіл кластерге, оқу базаңызға ғана емес, тиесілі. Жұмыс тұрақты атаулы `course_bi_reader_l3`, `course_region_east_l3`, `course_region_west_l3`, `course_orders_reader_l3`, `course_shared_reader_l3` рөлдерін, кездейсоқ атаулы уақытша пайдаланушыларды құрады; құпиясөздері жадыда, көрсетілмейді. Ұяшық сәтсіз болса да жояды. Екі нәрсені сақтайды: 6-бөлімнің `published_orders_l3`, `candidate_orders_l3` кестелері; егер 11-жұмыстың қосымша тіркелгісі бар болса, тақта жұмыс істей беруі үшін 5-бөлім көшіретін `course_bi_dashboard_l3` рөлі.

Хосты `'%'` жеке сынақ ортасына сай; өндірістік тіркелгіге жеке хост/тіркелгі деректерін жобалау керек. Root жаттығуды әкімші ретінде орындайды, тұтынушы оқуын тексеретін сәйкестік емес.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

import secrets

import pymysql

from dw_course.access import temporary_user
from dw_course.docker_runtime import connect_sandbox
from dw_course.operations import audit_records
from dw_course.runtime import expect
from dw_course.ui import card, show_records, show_sql

lab = connect_sandbox(module=12)


def roles():
    """SHOW ROLES, one dict per role, by name."""
    with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
        cursor.execute("SHOW ROLES")
        return {row["Name"]: row for row in cursor.fetchall()}


def grants(statement):
    """SHOW GRANTS or SHOW ALL GRANTS, one dict per account."""
    with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
        cursor.execute(statement)
        return cursor.fetchall()

<a id="unit-u.888de2b2a35740e084b54b5f23e583c5"></a>

<a id="1-check-the-identity-the-release-object-and-the-role-baseline"></a>

## 1. Сәйкестікті, жариялау нысанын, рөлдің бастапқы күйін тексеру

Өзгеріске дейін кім, қай нысанда жасайтынын, қазіргі күйін жазыңыз. Ұяшық алдымен бұрынғы орындаудан қалған рөлді жояды, сондықтан бастапқы күйі — сол тазалаудан кейінгі; өндірісте бірдеңені жоймас бұрын ескі күй жазылады. Кейін ағымдағы пайдаланушы/база мен рұқсат берушінің өз рұқсаттарын көрсетіп, курс рөлі жоқтығын, жариялау нысанын — семантикалық көріністің 2 күні, 10 тапсырысы, 1310.00 сомасын тексереді.

In [ ]:
lab.execute("DROP ROLE IF EXISTS course_bi_reader_l3")
lab.sql("SELECT CURRENT_USER() AS current_user, DATABASE() AS current_database", title="Who makes the change, and where")
lab.sql("SHOW GRANTS", title="The grantor's own privileges", columns=["UserIdentity", "Roles", "GlobalPrivs", "DatabasePrivs"])
expect("course_bi_reader_l3" in roles(), False, title="The course role does not exist yet")
expect(lab.query("SELECT COUNT(*), SUM(order_count), SUM(gross_amount) FROM bi_order_metrics_l2"), [(2, 10, "1310.00")],
       title="The release object: 2 days, 10 orders, 1310.00")

<a id="unit-u.8f20fdaf26444fc7b169031ff0fab063"></a>

**Күтілетін нәтиже**

Оқу базасында `'root'@'%'`; root рұқсаттары: `Node_priv,Admin_priv` бар `operator` рөлі, `information_schema` мен `mysql` оқу. Екі жасыл тексеру: курс рөлі әлі жоқ, көріністе 2 күн, 10 тапсырыс, 1310.00.

Root рөл құрып, кез келген рұқсат бере алады, сондықтан өзгерісті өзі орындайды. Тақтаға бұлар қажет емес; керегі келесіде рөл үстінде анықталады.

<a id="unit-u.03e75b83f669452f8a1c203a821cc6d7"></a>

<a id="2-grant-a-role-read-access-to-the-semantic-view-only"></a>

## 2. Рөлге тек семантикалық көріністі оқуға рұқсат беру

Doris рұқсатты рөлге, рөлді пайдаланушыға береді: BI бесеуі бөліседі, біреуі кетсе мүшелік қана өзгереді. Рөл оқу құқығы `SELECT_PRIV` мәнін бір нысанда, `internal.<your database>.bi_order_metrics_l2`, алады. Жаңа кестелері де оқылар базада емес, тұтынушы айналып өтуі мүмкін дереккөздерде де емес.

`SHOW ROLES` рөл рұқсаттарын ауқыммен көрсетеді. Екінші сұрау — ID, рөл, нысан, рұқсат, себебі бар өзгеріс жазбасының мысалы: өзгерісті құжаттайды, мақұлдау да, `GRANT` орындалды деген аудит те емес.

In [ ]:
grant_sql = f"GRANT SELECT_PRIV ON internal.{lab.database}.bi_order_metrics_l2 TO ROLE 'course_bi_reader_l3'"
show_sql("The role and its grant", "CREATE ROLE course_bi_reader_l3;\n" + grant_sql + ";")
lab.execute("CREATE ROLE course_bi_reader_l3")
lab.execute(grant_sql)
show_records("The role after the grant", [{key: roles()["course_bi_reader_l3"][key] for key in
                                           ("Name", "Users", "GlobalPrivs", "DatabasePrivs", "TablePrivs")}])
role = roles()["course_bi_reader_l3"]
expect((role["GlobalPrivs"], role["DatabasePrivs"], role["TablePrivs"]),
       (None, None, f"internal.{lab.database}.bi_order_metrics_l2: Select_priv"),
       title="The role can read the view, and nothing else")
lab.sql("""SELECT 'lab12-example' AS change_id, 'course_bi_reader_l3' AS role_name,
       'bi_order_metrics_l2' AS object_name, 'SELECT_PRIV' AS privilege,
       'course exercise: read-only dashboard role' AS change_reason""", title="An example change record, not an audit record");

<a id="unit-u.7fbc7736506d4089b105fd5e77a6cb0a"></a>

**Күтілетін нәтиже**

Екі команда, кейін рөл: әзірге мүшесіз, жаһандық/база рұқсатынсыз, `TablePrivs` — `internal.<your database>.bi_order_metrics_l2: Select_priv`. Жасыл тексеру, кейін бір жолдық өзгеріс жазбасы.

`GlobalPrivs`, `CatalogPrivs`, `DatabasePrivs` рұқсаты бір көріністен кең. `Select_priv` оқытады, бірақ рұқсат бергізбейді (`Grant_priv`), әкімшілік бермейді (`Admin_priv`).

<a id="unit-u.85ccfe8ce5524154974a32823d0c8a1c"></a>

<a id="3-test-the-role-as-an-ordinary-user"></a>

## 3. Рөлді қарапайым пайдаланушы ретінде тексеру

`SHOW ROLES` конфигурацияны, тек тұтынушы сұрауы нақты мүмкіндігін көрсетеді. `temporary_user()` кездейсоқ атау/құпиясөзбен қарапайым пайдаланушы құрып, рөл береді, блок соңында тексеру өтпесе де жояды. Сол пайдаланушы көріністі, екі дереккөзін, рөл қайтарылған соң қайта көріністі сұрайды. Төрт сұраудың қайсысына тыйым болатынын болжаңыз.

In [ ]:
answers = {}
with temporary_user(lab, "course_bi_reader_l3") as reader:
    expect(reader.query(f"SELECT CAST(order_date AS STRING), order_count, gross_amount FROM {lab.database}.bi_order_metrics_l2 ORDER BY order_date"),
           [("2026-01-01", 9, "1200.00"), ("2026-01-02", 1, "110.00")], title="The user reads the view")
    for source in ("daily_order_metrics_l2", "orders_fact_l2"):
        answers[f"{source} (a source of the view)"] = reader.denied(f"SELECT COUNT(*) FROM {lab.database}.{source}")
    expect(all(answers.values()), True, title="The user cannot read the view's sources")
    lab.execute(f"REVOKE 'course_bi_reader_l3' FROM {reader.identity}")
    answers["bi_order_metrics_l2, after REVOKE"] = reader.denied(f"SELECT COUNT(*) FROM {lab.database}.bi_order_metrics_l2")
    expect(answers["bi_order_metrics_l2, after REVOKE"] is not None, True, title="Without the role, the view is refused too")
show_records("What Doris answered the user", [{"query": query, "answer": answer} for query, answer in answers.items()])

<a id="unit-u.ffe7d6d994fe4387a9c2f96b9e2b8f72"></a>

**Күтілетін нәтиже**

Уақытша пайдаланушы құру командалары, кейін үш жасыл тексеру: көріністің екі жолын оқиды, екі дереккөзін оқымайды, `REVOKE` соң көріністі де оқымайды. Әр дереккөз жауабы: `Access denied; you need (at least one of) the (Admin_priv,Select_priv) privilege(s) on table daily_order_metrics_l2 for this operation` (не `orders_fact_l2`); `REVOKE` соң `Access denied for user 'course_reader_l3_<random>' to database '<your database>'`.

Дереккөздер оқылмаса да көрініс оқылады: Doris пайдаланушы атаған көрініс рұқсатын тексереді. Рөлсіз оқу базасында рұқсат қалмайды, сондықтан қосылудың өзіне тыйым салады. Ұяшық соңында пайдаланушы жойылады.

<a id="unit-u.d3f0760eef184b0fb2a944c3119be294"></a>

<a id="4-retire-the-access"></a>

## 4. Қолжетімділікті тоқтату

Тақта тоқтатылды, рөлде мүше қалмады: уақытша пайдаланушы жойылған. Рөлді осы ретпен екі команда тоқтатады: рұқсатын қайтарып, кейін жою. Көрініс пен жолдары өзгермейді.

In [ ]:
retire_sql = [f"REVOKE SELECT_PRIV ON internal.{lab.database}.bi_order_metrics_l2 FROM ROLE 'course_bi_reader_l3'",
              "DROP ROLE IF EXISTS course_bi_reader_l3"]
show_sql("Retire the role", ";\n".join(retire_sql) + ";")
for statement in retire_sql:
    lab.execute(statement)
expect("course_bi_reader_l3" in roles(), False, title="The role is gone")
expect(lab.query("SELECT COUNT(*), SUM(order_count), SUM(gross_amount) FROM bi_order_metrics_l2"), [(2, 10, "1310.00")],
       title="The view is unchanged")

<a id="unit-u.d240bb93f460472898fb72433fa2e98e"></a>

**Күтілетін нәтиже**

Екі команда, екі жасыл тексеру: рөл жоқ, көріністе әлі 2 күн, 10 тапсырыс, 1310.00.

Басқа тақта рөлді қолданса, рұқсатты қайтару оны да ажыратар еді; жаттығу ортақ рөлдегі екеудің бірін тоқтатады. Тоқтату мақұлдауы мен өзгеріс жазбасын сақтаңыз: `DROP ROLE` нысанды жояды, себебін жазбайды.

<a id="unit-u.a969286464614d17b7bc2a2fca0d7320"></a>

<a id="5-move-a-direct-grant-to-a-role"></a>

## 5. Тікелей рұқсатты рөлге көшіру

11-модуль тақтасы Doris тіркелгісін құрып, төлем көрінісін тікелей пайдаланушыға берген. Бұл істейді, бірақ рөл тексеруінде көрінбейді; сол тақтаның келесі тіркелгісіне қайта беру қажет.

`SHOW ALL GRANTS` әр тіркелгінің өз және рөлдік рұқсаттарын біріктіріп береді, қайсысы қайдан екенін айтпайды. Рөлсізінде тек өзінікі бар. Ұяшық оқу базаңыздың кесте рұқсаттары бар рөлсіз тіркелгілерді тізеді. Тақтаның `course_bi_<suffix>` тіркелгісін тапса, `course_bi_dashboard_l3` рөліне көшіреді: рөлге рұқсат, тіркелгіге рөл беру, кейін өз рұқсатын қайтару. Осы ретте қатынау үзілмейді, тақта жұмыс істейді.

Doris өз рұқсаттарын `default_role_rbac_<user>@<host>` жасырын рөлде сақтайды. Онда жоқ рұқсатты қайтару сол рөлді атап сәтсіз болады; екінші `REVOKE` тікелей рұқсат қалмағанын дәлелдейді. Тақта құрмасаңыз, көшіретін ештеңе жоқ деп айтады.

In [ ]:
database_grants = [row for row in grants("SHOW ALL GRANTS")
                   if f"internal.{lab.database}." in (row["TablePrivs"] or "")]
direct = [row for row in database_grants if not row["Roles"]]
dashboard_accounts = [row for row in database_grants if row["UserIdentity"].startswith("'course_bi_")]
show_records("Accounts without roles that hold privileges in this database",
             [{"account": row["UserIdentity"], "table_privileges": row["TablePrivs"]} for row in direct],
             columns=["account", "table_privileges"])
dashboard = [row["UserIdentity"] for row in direct if row["UserIdentity"].startswith("'course_bi_")]
if dashboard:
    account = dashboard[0]
    revoke_direct = f"REVOKE SELECT_PRIV ON internal.{lab.database}.bi_paid_kpis_l2 FROM {account}"
    move_sql = [
        "CREATE ROLE IF NOT EXISTS course_bi_dashboard_l3",
        f"GRANT SELECT_PRIV ON internal.{lab.database}.bi_paid_kpis_l2 TO ROLE 'course_bi_dashboard_l3'",
        f"GRANT 'course_bi_dashboard_l3' TO {account}",
        revoke_direct,
    ]
    show_sql("Move the dashboard account's grant to a role", ";\n".join(move_sql) + ";")
    for statement in move_sql:
        lab.execute(statement)
    after = grants(f"SHOW GRANTS FOR {account}")[0]
    show_records("The dashboard account after the move", [{key: after[key] for key in ("UserIdentity", "Roles", "TablePrivs")}])
    try:
        lab.execute(revoke_direct)
        left = "A direct grant was still there; this REVOKE removed it"
    except pymysql.err.OperationalError as error:
        left = None if "does not exist in role default_role_rbac" in str(error) else str(error)
    expect((after["Roles"], left), ("course_bi_dashboard_l3", None),
           title="The account reads through the role, and no direct grant is left")
elif any("course_bi_dashboard_l3" in (row["Roles"] or "").split(",") for row in dashboard_accounts):
    show_records("The dashboard already reads through its role",
                 [{key: row[key] for key in ("UserIdentity", "Roles", "TablePrivs")} for row in dashboard_accounts])
    card("The dashboard account already reads through course_bi_dashboard_l3. No grants were changed. "
         "Refresh the saved Superset charts to check access.", "ok", "Already migrated")
elif dashboard_accounts:
    show_records("Inspect the dashboard account's existing roles before changing grants", dashboard_accounts)
    card("A dashboard account with roles already exists in this database. No grants were changed; "
         "inspect its roles and TablePrivs above before changing access.", "skip", "Review existing roles")
else:
    card("No course dashboard account has table privileges in this database. Module 11's optional "
         "dashboard was not built here, so there is nothing to move.", "skip", "No dashboard account")

<a id="unit-u.8ba3b9b9c322484e9edd0f12138c87e2"></a>

**Күтілетін нәтиже**

11-модуль тақтасын құрсаңыз: `internal.<your database>.bi_paid_kpis_l2: Select_priv` бар `'course_bi_<suffix>'@'%'` бір тіркелгісі; төрт команда; кейін `course_bi_dashboard_l3` рөлі бар, `TablePrivs` ішінде енді рөлден келген `bi_paid_kpis_l2: Select_priv` сақталған тіркелгі. Жасыл тексеру: екінші `REVOKE` `... does not exist in role default_role_rbac_course_bi_<suffix>@%` қатесімен тоқтады, тікелей рұқсат жоқ. Superset тақтасын жаңартыңыз: диаграммалар енді рөл арқылы ашылады. Қайталағанда тікелей рұқсат кестесі бос, тіркелгіде `course_bi_dashboard_l3` әлі бар; **Already migrated** карточкасы еш рұқсат өзгермегенін айтады. Сақталған тақта жұмыс істейді, қайта құрмаңыз.

Қосымша тақтаны осы базада құрмасаңыз: бос кесте, сұр **No dashboard account** карточкасы. Өзге рөлдері бар тіркелгі бөлек қарауға көрсетіледі, ұяшық оларды өзгертпейді.

Рұқсаты үзілген жоқ: өзінікі қайтарылмас бұрын рөл алған. Алдымен қайтарса, тақта аралықта істемес еді.

<a id="unit-u.58442c917bd24c759c4a23e8304a1054"></a>

<a id="6-validate-cut-over-and-roll-back-a-table-release"></a>

## 6. Кесте жариялауын тексеру, ауыстыру және кері қайтару

Атауды алмастыру атомарлы, дұрыс жариялау — кеңірек рәсім. Бұл басқарылатын жаттығу бір келісімшартты ескі/үміткер кестелерді құрып, үміткерді толтырады, **кідіртілген жазушыны** имитациялап, соңғы өзгерісін екеуіне қолданып, әр бизнес жолын жариялауға дейін салыстырады.

`swap=true` ескіні үміткер атымен сақтайды. Алдымен жазушылар тоқтағанда кері қайтаруды жаттығыңыз. Кейін қайта жариялап, бір имитациялық жазуды жалғастырыңыз. **Болжаңыз:** ескі енді қауіпсіз кері қайтару нысанасы ма? Жоқ: жаңа жазуы жоқ. Мұнда нақты CDC конвейері кідіртілмейді; бұл үзіліссіз көшіруді емес, ауысу шекарасы талабын көрсетеді.

In [ ]:
for table in ("published_orders_l3", "candidate_orders_l3"):
    lab.execute(f"DROP TABLE IF EXISTS {table}")
    lab.execute(f"""CREATE TABLE {table} (
        order_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
    ) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
    PROPERTIES ("replication_num"="1")""")
lab.insert("published_orders_l3", ["order_id","amount"], [(7101,70),(7102,90)])
lab.execute("INSERT INTO candidate_orders_l3 SELECT * FROM published_orders_l3")
# Simulated final delta at a paused-writer boundary, applied before validation.
for table in ("published_orders_l3", "candidate_orders_l3"):
    lab.insert(table, ["order_id","amount"], [(7102,95),(7103,40)])
release_rows = [(7101,"70.00"),(7102,"95.00"),(7103,"40.00")]
expect(lab.query("SELECT * FROM candidate_orders_l3 ORDER BY order_id"), release_rows, title="Candidate reconciles against the final-delta ledger")
expect(lab.query("SELECT * FROM published_orders_l3 ORDER BY order_id"), release_rows, title="The published table holds the same final delta")
lab.execute('ALTER TABLE published_orders_l3 REPLACE WITH TABLE candidate_orders_l3 PROPERTIES("swap"="true")')
expect(lab.query("SELECT * FROM published_orders_l3 ORDER BY order_id"), release_rows, title="Stable consumer name after release")
# Rollback rehearsal BEFORE writers resume: both sides still contain the validated cutoff.
lab.execute('ALTER TABLE published_orders_l3 REPLACE WITH TABLE candidate_orders_l3 PROPERTIES("swap"="true")')
expect(lab.query("SELECT * FROM published_orders_l3 ORDER BY order_id"), release_rows, title="Paused-writer rollback reconciles")
lab.execute('ALTER TABLE published_orders_l3 REPLACE WITH TABLE candidate_orders_l3 PROPERTIES("swap"="true")')
lab.insert("published_orders_l3", ["order_id","amount"], [(7104,25)])
expect(lab.query("SELECT COUNT(*),SUM(amount) FROM published_orders_l3"), [(4,"230.00")])
expect(lab.query("SELECT COUNT(*),SUM(amount) FROM candidate_orders_l3"), [(3,"205.00")], title="Retained old data lacks the post-release write; do not swap it back blindly")
lab.sql("SELECT 'current publication' AS data_copy, COUNT(*) AS orders,SUM(amount) AS amount FROM published_orders_l3 UNION ALL SELECT 'retained cutoff',COUNT(*),SUM(amount) FROM candidate_orders_l3", title="Rollback safety depends on the write boundary");

<a id="unit-u.8055cbc32a8a4997850fd5457c8c46bf"></a>

**Күтілетін нәтиже**

Бес жасыл тексеру: үміткер соңғы тізілімдегі 7101 (70.00), 7102 (95.00), 7103 (40.00) мәндеріне сәйкес; жарияланған кесте де; атауы жариялағаннан кейін жауап береді; жазушылар әлі тоқтағанда кері қайтару сол жолдарды береді; кейінгі бір жазу сақталған ескіде жоқ. Соңғы кесте: ағымдағыда 4 тапсырыс, 230.00; ескіде 3, 205.00.

Алмастыру атомарлы, оқырман жарты кесте көрмейді. Айналасындағы рәсім қауіпсіз етеді: кідірген жазушы, салыстырылған соңғы өзгеріс және жариялағаннан бері жазу болмағанша ғана қауіпсіз кері қайтару.

<a id="unit-u.70bfd5125add41f496daed81279ff4b2"></a>

<a id="7-limit-each-region-to-its-own-orders-with-row-policies"></a>

## 7. Row Policy арқылы аймақты өз тапсырыстарымен шектеу

EAST тек EAST, WEST тек WEST тапсырыстарын оқиды. Кесте рұқсаты мұны айта алмайды: барлық жолды қамтиды. **Row Policy** аталған пайдаланушы/рөлдің әр сұрауына шарт қосады; `RESTRICTIVE` сәйкестігін ғана қалдырады. Әр аймақ `orders_fact_l2` үстінде `SELECT_PRIV` бар рөл және сол рөл саясатын алады; рұқсат сияқты саясат мүшелеріне ілеседі.

Әр аймақтың уақытша пайдаланушысы ретінде ұяшық аймақпен тапсырыс оқиды, әдейі өзгесін сұрайды, 9-бөлім аудиттен табатын белгіленген сұрау орындайды. Root ретінде тексеру ештеңе дәлелдемейді: саясат root-қа қолданылмайды.

In [ ]:
run_tag = "course_audit_" + secrets.token_hex(6)
audited_users = []
regions = {"EAST": "course_region_east_l3", "WEST": "course_region_west_l3"}
policy_sql = []
for region, role in regions.items():
    policy_sql += [f"CREATE ROLE {role}",
                   f"GRANT SELECT_PRIV ON internal.{lab.database}.orders_fact_l2 TO ROLE '{role}'",
                   f"CREATE ROW POLICY region_{region.lower()}_l3 ON {lab.database}.orders_fact_l2 "
                   f"AS RESTRICTIVE TO ROLE {role} USING (region = '{region}')"]
show_sql("Two roles, each with a Row Policy", ";\n".join(policy_sql) + ";")


def drop_region_objects():
    # A dropped role leaves its Row Policies behind, so drop the policies first.
    for region, role in regions.items():
        lab.execute(f"DROP ROW POLICY IF EXISTS region_{region.lower()}_l3 ON {lab.database}.orders_fact_l2")
        lab.execute(f"DROP ROLE IF EXISTS {role}")


drop_region_objects()
visible = []
try:
    for statement in policy_sql:
        lab.execute(statement)
    for region, role in regions.items():
        with temporary_user(lab, role, prefix="course_region_l3") as reader:
            rows = reader.query("SELECT region, COUNT(*), SUM(order_amount) FROM orders_fact_l2 GROUP BY region")
            other = "WEST" if region == "EAST" else "EAST"
            expect((rows, reader.query("SELECT COUNT(*) FROM orders_fact_l2 WHERE region = %s", (other,))),
                   ([(region, 6, "830.00")] if region == "EAST" else [(region, 5, "680.00")], [(0,)]),
                   title=f"The {region} user sees only {region} orders, even when it asks for {other}")
            reader.query(f"SELECT '{run_tag}_{region}' AS request_tag, COUNT(*) AS orders FROM orders_fact_l2")
            audited_users.append(reader.name)
            visible.append({"user": reader.name, "role": role, "region": rows[0][0], "orders": rows[0][1], "amount": str(rows[0][2])})
finally:
    drop_region_objects()
show_records("What each regional user could see", visible)
expect([row[0] for row in lab.query("SHOW ROW POLICY") if row[2] == lab.database and row[3] == "orders_fact_l2"], [],
       title="No Row Policy is left on orders_fact_l2")

<a id="unit-u.9fbe57a9ec2c434ca7ed01c46aaf5996"></a>

**Күтілетін нәтиже**

Командалар, кейін екі жасыл тексеру: EAST 6 тапсырыс/830.00, WEST-тен ештеңе көрмейді; WEST 5/680.00, EAST-тен ештеңе көрмейді. Көргендері кестесі, соңғы жасыл тексеру: `orders_fact_l2` үстінде саясат қалмады.

EAST пайдаланушысының `WHERE region = 'WEST'` сұрауы 0 қайтарды: саясат базада қолданылады, тақта сүзгісінен өзгеше, еш сұрау кеңейте алмайды. Рөлді жою саясаттарын жоймайтындықтан ұяшық алдымен саясатты жояды: кейін сол атпен құрылған рөл қайта алар еді.

<a id="unit-u.016d385286204f13a5a95be4ae8d9269"></a>

<a id="8-keep-the-payment-columns-from-a-role-with-column-privileges"></a>

## 8. Баған рұқсатымен төлем бағандарын рөлден жасыру

Төленген/қайтарылғанды тек Қаржы көреді. Қалғанына рөл `orders_fact_l2` бес бағанында `SELECT_PRIV` алады: `order_id`, `order_date`, `status`, `order_amount`, `region`. Пайдаланушысы ретінде ұяшық оларды оқып, `paid_amount`, `SELECT *`, `COUNT(*)` сұрайды. Қайсысы қабылданбайтынын болжаңыз. Алғашқы екеуінде 9-бөлім белгісі бар: бірі орындалады, екіншісіне тыйым салынады.

In [ ]:
column_sql = (f"GRANT SELECT_PRIV(order_id, order_date, status, order_amount, region) "
              f"ON internal.{lab.database}.orders_fact_l2 TO ROLE 'course_orders_reader_l3'")
show_sql("A role with five columns", "CREATE ROLE course_orders_reader_l3;\n" + column_sql + ";")
lab.execute("DROP ROLE IF EXISTS course_orders_reader_l3")
answers = {}
try:
    lab.execute("CREATE ROLE course_orders_reader_l3")
    lab.execute(column_sql)
    with temporary_user(lab, "course_orders_reader_l3", prefix="course_orders_l3") as reader:
        expect(reader.query(f"SELECT '{run_tag}_columns' AS request_tag, COUNT(order_id), SUM(order_amount) FROM orders_fact_l2"),
               [(f"{run_tag}_columns", 11, "1510.00")], title="The user reads the five columns")
        answers["SUM(paid_amount)"] = reader.denied(f"SELECT '{run_tag}_payments' AS request_tag, SUM(paid_amount) FROM orders_fact_l2")
        answers["SELECT *"] = reader.denied("SELECT * FROM orders_fact_l2")
        answers["COUNT(*)"] = reader.denied("SELECT COUNT(*) FROM orders_fact_l2")
        expect(all(answers.values()), True, title="The payment column, SELECT *, and COUNT(*) are refused")
        audited_users.append(reader.name)
finally:
    lab.execute("DROP ROLE IF EXISTS course_orders_reader_l3")
show_records("What Doris answered the user", [{"query": query, "answer": answer} for query, answer in answers.items()])

<a id="unit-u.afd81f07fa3e4f118bad4b90e02edac1"></a>

**Күтілетін нәтиже**

Екі жасыл тексеру: пайдаланушы өз бағандарынан 11 тапсырыс, 1510.00 оқиды; Doris басқа үшеуін қабылдамайды. `SUM(paid_amount)` үшін: `Permission denied: user [...] does not have privilege for [priv predicate: OR, Admin_priv,Select_priv] command on [internal].[<your database>].[orders_fact_l2].[paid_amount]`; `SELECT *` үшін сол қате `[customer_id]` бойынша; `COUNT(*)` үшін `Access denied; you need (at least one of) the (Admin_priv,Select_priv) privilege(s) on table orders_fact_l2 for this operation`.

`SELECT *` әр бағанды сұрайды, Doris рөлде жоқ алғашқы `customer_id` бағанын атайды. `COUNT(*)` баған оқымайды, бірақ бүкіл кесте рұқсатын талап етеді; `COUNT(order_id)` берілген бағанмен сол жолдарды санайды. Баған рұқсаты қазіргі бағандарды қамтиды, кейін қосылғаны кірмейді: жаңа деректі әдепкіде көрмеуі тиіс рөлге сай, көруі қажет болса қайта қарау керек.

<a id="unit-u.36777a43f49842debcd407b7c3b36d45"></a>

<a id="9-find-the-users-queries-in-the-audit-table"></a>

## 9. Пайдаланушы сұрауларын аудиттен табу

Рұқсат сәйкестік не істей алатынын, жоғарғы нәтиже не көргенін, аудит шын не орындалғанын айтады. Плагин әр команданы пайдаланушы, база, күй, қате кодымен SQL оқитын `__internal_schema.audit_log` ішіне жазады. Кемінде минутына бір рет пакетпен жазады, сондықтан ұяшық уақытша пайдаланушылардың төрт белгіленген сұрауын күтеді: екі аймақ, бес баған және `paid_amount` тыйым сұрауы.

Пайдаланушы сүзгісі root командаларын және root тергенді алып тастайды. Сол жазбалар FE `fe.audit.log` файлында да бар.

In [ ]:
audit_records(lab, audited_users, run_tag, expected=4);

<a id="unit-u.4b6fb7605e504521a4642b477a297609"></a>

**Күтілетін нәтиже**

Бір минутқа дейінгі барыс карточкасы, кейін төрт жол — белгіленген әр сұрауға бір жол: екі аймақтың `SELECT '<tag>_EAST' ...`, `SELECT '<tag>_WEST' ...`, бес баған пайдаланушысының `SELECT '<tag>_columns' ...`; әрқайсысында `state` EOF, `error_code` 0, `return_rows` 1. Оның `SELECT '<tag>_payments' ...` сұрауы — `state` ERR, `error_code` 1105, `return_rows` 0. Әр жолда пайдаланушы, оқу базасы, сұрау ID-і бар.

EOF — жол қайтарған күй; ERR қате кодымен — тыйым не басқа қате. Жазба кім не орындап, Doris не жауап бергенін айтады. Саясаттар дұрыс па, барлық FE жазбасы жиналды ма, тіркелгінің артында кім тұрды — дәлелдемейді; әр FE өз жазбасын жазады.

<a id="unit-u.92ed597a9a4d43f99e078688b6ffd312"></a>

<a id="10-practice-retire-one-of-two-dashboards-that-share-a-role"></a>

## 10. Жаттығу: ортақ рөлдегі екі тақтаның бірін тоқтату

A және B `bi_order_metrics_l2` көрінісін бір рөлмен оқиды. A тоқтайды, B қалады. Көрініске `SELECT_PRIV` бар `course_shared_reader_l3` рөлін құрыңыз; әр тақтаға бірден, барлығы екі уақытша пайдаланушы құрыңыз. Екеуі оқитынын тексеріп, тек A-ны тоқтатыңыз; A-ға тыйым, B әлі оқитынын растаңыз. Соңында, тексеру сәтсіз болса да, рөлді жойыңыз.

In [ ]:
# Your code here

<a id="unit-u.696da6f2962741739fa3331da07e3f66"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
shared_grant = f"GRANT SELECT_PRIV ON internal.{lab.database}.bi_order_metrics_l2 TO ROLE 'course_shared_reader_l3'"
view = f"SELECT SUM(order_count) FROM {lab.database}.bi_order_metrics_l2"
lab.execute("DROP ROLE IF EXISTS course_shared_reader_l3")
try:
    lab.execute("CREATE ROLE course_shared_reader_l3")
    lab.execute(shared_grant)
    with temporary_user(lab, "course_shared_reader_l3", prefix="course_dashboard_a") as a, \
         temporary_user(lab, "course_shared_reader_l3", prefix="course_dashboard_b") as b:
        expect((a.query(view), b.query(view)), ([(10,)], [(10,)]), title="Both dashboards read the view")
        lab.execute(f"REVOKE 'course_shared_reader_l3' FROM {a.identity}")
        expect((a.denied(view) is not None, b.query(view)), (True, [(10,)]),
               title="A is refused; B still reads through the role")
finally:
    lab.execute("DROP ROLE IF EXISTS course_shared_reader_l3")

<a id="unit-u.7ecadd5bc8fa4461a07a709673cb3226"></a>

**Күтілетін нәтиже**

Екі пайдаланушыны құру командалары, кейін екі жасыл тексеру: екеуі 10 тапсырыс оқиды; A мүшелігінің `REVOKE` соң A-ға тыйым, B әлі оқиды.

Рөл рұқсатын қайтару B-ні де ажыратар еді. Тоқтатпас бұрын `SHOW ROLES` ішіндегі рөл `Users` тізімін және басқа жолдарды қараңыз: тікелей рұқсат не басқа рөл A-ға оқыта беруі мүмкін.

<a id="unit-u.0d1a7065caa64875bc34178a1b45875d"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- Тек `bi_order_metrics_l2` үстіндегі `SELECT_PRIV` рөлі қарапайым пайдаланушыға көріністі оқытты, дереккөзін оқытпады; `SHOW ROLES`, пайдаланушы сұраулары, өзгеріс жазбасы нені көрсететінін айта аласыз;
- мүшелікті/рөл рұқсатын қашан қайтаруды, рөлді қандай ретпен тоқтатуды білесіз;
- тіркелгі қатынауын үзбей тікелей рұқсатты рөлге көшіре аласыз;
- кестені кері қайтару неге келесі жазуға дейін ғана қауіпсіз екенін айта аласыз;
- Row Policy мен тақта сүзгісін ажыратып, баған рұқсатында `COUNT(*)` неге өтпегенін түсіндіре аласыз;
- сұраулар мен тыйымды `__internal_schema.audit_log` ішінен тауып, аудит нені дәлелдемейтінін айта аласыз.

<a id="unit-u.dc599d8ab7c640a0888a364e9e9bad5b"></a>

<a id="next"></a>

## Әрі қарай

[12-тест: Жариялау, рұқсаттар және аудит](quiz12_publishing_permissions_audit.ipynb) файлын ашып, [13-модуль: Сақтау мен өмірлік циклді басқару](../module13-storage-lifecycle/course.md) бөліміне өтіңіз.